# 01 Prepare the survey data

This is the only notebook that opens the raw Qualtrics exports. It links each person's first and second survey, gives everyone a study ID, and writes de-identified files to `data/processed`.

Names, contact details, IP addresses, locations, dates and free-text answers are dropped here and never reach the analysis notebook. The raw files are not part of this repository; `data/README.md` lists what goes in `data/raw`.

In [1]:
import difflib
import json
import random
import re
import unicodedata
from pathlib import Path

import pandas as pd

# notebooks sit one level below the project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

## Reading the exports

Every Qualtrics export carries two header rows, the question text and the variable names. The survey files have the variable names second and the advisory board export has them first, so the header row is set per file. File names vary slightly between exports, so each one is found by a few key words.

In [2]:
def find_file(*words, exclude=()):
    hits = []
    for path in RAW.glob("*.xlsx"):
        name = path.name.lower()
        if all(w in name for w in words) and not any(x in name for x in exclude):
            hits.append(path)
    assert len(hits) == 1, f"expected one file for {words}, found {[h.name for h in hits]}"
    return hits[0]


def read_export(path, names_row=1):
    df = pd.read_excel(path, sheet_name="Sheet0", header=names_row)
    if names_row == 0:
        # here the question text sits under the variable names, so drop it
        df = df.iloc[1:]
    return df.dropna(how="all").reset_index(drop=True)


skip = ("summary", "figure")
surveys = {
    "child_first": read_export(find_file("5-12", "pre", exclude=skip)),
    "child_second": read_export(find_file("5-12", "post", exclude=skip)),
    "adult_first": read_export(find_file("13", "pre", exclude=skip + ("5-12",))),
    "adult_second": read_export(find_file("13", "post", exclude=skip + ("5-12",))),
}
board = read_export(find_file("advisory", "demographic"), names_row=0)

for key, df in surveys.items():
    print(f"{key:13} {len(df):3} surveys")
print(f"{'board':13} {len(board):3} members")

assert {k: len(v) for k, v in surveys.items()} == {
    "child_first": 20, "child_second": 10, "adult_first": 83, "adult_second": 39}
assert len(board) == 11

child_first    20 surveys
child_second   10 surveys
adult_first    83 surveys
adult_second   39 surveys
board          11 members


## Matching names

People typed their own names, so the same person can show up as a first name on one survey and a full name on the next, or with and without an accent. Each second survey is linked to at most one first survey, trying these in order:

1. the same name once case, accents, spaces and punctuation are ignored
2. one name is the start of the other, such as a first name alone and a full name
3. a very close spelling
4. the names share a first name or surname, age (within two years), gender and race agree, and only one first survey fits

Steps 2 to 4 also require gender and age to agree whenever both surveys recorded them.

In [3]:
def plain(value):
    # drop accents so a name typed with and without them compares as equal
    text = unicodedata.normalize("NFKD", str(value))
    return text.encode("ascii", "ignore").decode().lower()


def name_key(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", plain(value))


def name_parts(value):
    if pd.isna(value):
        return set()
    return {part for part in re.split(r"[^a-z]+", plain(value)) if len(part) >= 3}


def first_race(value):
    # race was a check-all question, so keep the first category chosen
    if pd.isna(value) or str(value).strip() == "":
        return None
    return str(value).split(",")[0].strip()


def profile(row, prefix):
    age = pd.to_numeric(row.get(f"{prefix}_age"), errors="coerce")
    gender = row.get(f"{prefix}_gender")
    return {
        "age": None if pd.isna(age) else float(age),
        "gender": None if pd.isna(gender) else str(gender),
        "race": first_race(row.get(f"{prefix}_race_eth")),
    }


def plausible(a, b, check_race=False):
    # a missing value never counts against a match
    if a["gender"] and b["gender"] and a["gender"] != b["gender"]:
        return False
    if a["age"] is not None and b["age"] is not None and abs(a["age"] - b["age"]) > 2:
        return False
    if check_race and a["race"] and b["race"]:
        if not (a["race"].startswith("Not listed") or b["race"].startswith("Not listed")):
            return a["race"] == b["race"]
    return True


def link(first, second, first_col, second_col, first_prefix, second_prefix):
    keys = {i: name_key(v) for i, v in first[first_col].items()}
    exact = {}
    for i, k in keys.items():
        if k:
            exact.setdefault(k, i)
    profiles = {i: profile(first.loc[i], first_prefix) for i in first.index}

    links = {}
    for j, row in second.iterrows():
        n = name_key(row[second_col])
        if not n:
            continue
        mine = profile(row, second_prefix)
        match = None

        if n in exact:
            match = (exact[n], "exact")

        if match is None:
            hits = [i for i, k in keys.items()
                    if k and (k.startswith(n) or n.startswith(k)) and plausible(profiles[i], mine)]
            if len(hits) == 1:
                match = (hits[0], "first name or full name")

        if match is None:
            close = difflib.get_close_matches(n, list(exact), n=1, cutoff=0.86)
            if close and plausible(profiles[exact[close[0]]], mine):
                match = (exact[close[0]], "close spelling")

        if match is None:
            parts = name_parts(row[second_col])
            hits = [i for i in first.index
                    if parts & name_parts(first.loc[i, first_col])
                    and plausible(profiles[i], mine, check_race=True)]
            if len(hits) == 1:
                match = (hits[0], "name variant")

        if match is not None:
            links[j] = match
    return links

## Linking the two administrations

In [4]:
child_links = link(surveys["child_first"], surveys["child_second"],
                   "child_name", "child_name", "pre", "post")
adult_links = link(surveys["adult_first"], surveys["adult_second"],
                   "name", "name", "pre", "post")

for label, links, total in [("child", child_links, len(surveys["child_second"])),
                            ("adolescent and adult", adult_links, len(surveys["adult_second"]))]:
    methods = pd.Series([m for _, m in links.values()]).value_counts()
    print(f"{label}: {len(links)} of {total} second surveys linked on the same form")
    print("  " + ", ".join(f"{m}: {c}" for m, c in methods.items()))

assert len(child_links) == 10
assert len(adult_links) == 37

child: 10 of 10 second surveys linked on the same form
  exact: 10
adolescent and adult: 37 of 39 second surveys linked on the same form
  exact: 28, first name or full name: 5, name variant: 4


Two adolescent and adult second surveys are left. Both came from children who had completed the child survey first, so they are linked to that child but not paired, because the child survey was answered by a parent and the second one by the child.

In [5]:
leftover = surveys["adult_second"].drop(index=list(adult_links))
cross_form = link(surveys["child_first"], leftover, "child_name", "name", "pre", "post")

print(f"adolescent and adult second surveys from children who did the child survey first: {len(cross_form)}")
assert len(cross_form) == len(leftover) == 2

adolescent and adult second surveys from children who did the child survey first: 2


## Who took part

Two people completed both a child survey and an adolescent and adult survey at the first administration, so they are counted once.

In [6]:
same_person = link(surveys["child_first"], surveys["adult_first"],
                   "child_name", "name", "pre", "pre")

n_unique = len(surveys["child_first"]) + len(surveys["adult_first"]) - len(same_person)
print(f"completed both first surveys: {len(same_person)}")
print(f"unique participants: {n_unique}")
assert len(same_person) == 2 and n_unique == 101

completed both first surveys: 2
unique participants: 101


## Families

Households are counted from the parent or guardian named on each child survey, treating a first name on one child's survey and a full name on a sibling's as the same parent. Parents who also completed an adolescent and adult survey are counted only on an identical name, so that number is a floor.

In [7]:
parent_keys = [name_key(p) for p in surveys["child_first"]["parent_name"]]
households = []
for idx, k in enumerate(parent_keys):
    for group in households:
        if k and any(parent_keys[x] and (k.startswith(parent_keys[x]) or parent_keys[x].startswith(k))
                     for x in group):
            group.append(idx)
            break
    else:
        households.append([idx])

adult_keys = {name_key(v) for key in ("adult_first", "adult_second") for v in surveys[key]["name"]}
n_parents = len(households)
n_two_children = sum(len(g) > 1 for g in households)
n_parent_adult = sum(any(parent_keys[x] in adult_keys for x in g if parent_keys[x]) for g in households)

print(f"parents or guardians behind the {len(parent_keys)} child surveys: {n_parents}")
print(f"parents who enrolled two children: {n_two_children}")
print(f"parents who also completed an adolescent and adult survey (identical name): {n_parent_adult}")

parents or guardians behind the 20 child surveys: 17
parents who enrolled two children: 3
parents who also completed an adolescent and adult survey (identical name): 5


## Advisory board members among the participants

Several board members, including instructors, also filled in participant surveys. They are flagged here so the analysis can be run with and without them.

In [8]:
board_keys = {name_key(v) for v in board["name"] if name_key(v)}
respondent_keys = {name_key(v) for key in ("adult_first", "adult_second") for v in surveys[key]["name"]}
n_board_respondents = len(board_keys & respondent_keys)
print(f"board members who completed an adolescent and adult survey: {n_board_respondents} of {len(board_keys)}")

board members who completed an adolescent and adult survey: 7 of 11


## Study IDs

Linkage is finished, so names are no longer needed. Each person gets a study ID from P001 upward, handed out in a shuffled order so the number says nothing about when or where someone enrolled.

In [9]:
person_of = {}
next_person = 0
for i in surveys["child_first"].index:
    person_of[("child_first", i)] = next_person
    next_person += 1
for i in surveys["adult_first"].index:
    if i in same_person:
        person_of[("adult_first", i)] = person_of[("child_first", same_person[i][0])]
    else:
        person_of[("adult_first", i)] = next_person
        next_person += 1
assert next_person == n_unique

order = list(range(next_person))
random.Random(2024).shuffle(order)
study_id = {person: f"P{rank + 1:03d}" for rank, person in enumerate(order)}

ids = {
    "child_first": [study_id[person_of[("child_first", i)]] for i in surveys["child_first"].index],
    "adult_first": [study_id[person_of[("adult_first", i)]] for i in surveys["adult_first"].index],
    "child_second": [study_id[person_of[("child_first", child_links[j][0])]]
                     for j in surveys["child_second"].index],
}
ids["adult_second"] = [
    study_id[person_of[("adult_first", adult_links[j][0])]] if j in adult_links
    else study_id[person_of[("child_first", cross_form[j][0])]]
    for j in surveys["adult_second"].index
]

## Keeping only what the analysis needs

The processed files are built from a short list of columns rather than by deleting the ones known to be identifying, so nothing unexpected can slip through. Exact ages become the age bands used in Table 1, and dates become the number of days between a person's two surveys.

In [10]:
INCOME = {"child_first": "pre_parent_income", "child_second": "post_parent_income",
          "adult_first": "pre_income", "adult_second": "post_income"}
AGE_BANDS = [(5, "5 years"), (12, "6 to 12 years"), (17, "13 to 17 years"), (29, "18 to 29 years"),
             (39, "30 to 39 years"), (49, "40 to 49 years"), (59, "50 to 59 years")]


def age_band(value):
    age = pd.to_numeric(value, errors="coerce")
    if pd.isna(age):
        return "No response"
    for upper, label in AGE_BANDS:
        if age <= upper:
            return label
    return "60 years and older"


def calendar_days(first_date, second_date):
    start = pd.to_datetime(first_date).normalize()
    end = pd.to_datetime(second_date).normalize()
    return int((end - start).days)


ages = {key: pd.to_numeric(df[f"{'pre' if key.endswith('first') else 'post'}_age"], errors="coerce")
        for key, df in surveys.items()}
assert ages["child_first"].min() >= 5, "the lowest age band assumes nobody under 5"

processed = {}
for key, df in surveys.items():
    prefix = "pre" if key.endswith("first") else "post"
    out = pd.DataFrame({
        "study_id": ids[key],
        "form": "child" if key.startswith("child") else "adolescent_adult",
        "administration": "first" if key.endswith("first") else "second",
        "finished": df["Finished"].astype(str).str.strip().str.lower().eq("true").to_numpy(),
        "site": df["game_site"].to_numpy(),
    })
    for k in range(1, 10):
        out[f"item_{k}"] = df[f"{prefix}_game_impact_{k}"].to_numpy()
    out["age_band"] = df[f"{prefix}_age"].map(age_band).to_numpy()
    out["gender"] = df[f"{prefix}_gender"].to_numpy()
    out["race"] = df[f"{prefix}_race_eth"].map(first_race).to_numpy()
    out["race_more_than_one"] = df[f"{prefix}_race_eth"].fillna("").astype(str).str.contains(",").to_numpy()
    out["income"] = df[INCOME[key]].to_numpy()
    if key.startswith("adult"):
        out["on_advisory_board"] = df["name"].map(lambda v: name_key(v) in board_keys).to_numpy()
    else:
        out["on_advisory_board"] = False
    processed[key] = out

for key, first_key, links in [("child_second", "child_first", child_links),
                              ("adult_second", "adult_first", adult_links)]:
    second, first = surveys[key], surveys[first_key]
    out = processed[key]
    out["linked_same_form"] = [j in links for j in second.index]
    out["link_method"] = [links[j][1] if j in links else None for j in second.index]
    out["days_since_first"] = [
        calendar_days(first.loc[links[j][0], "StartDate"], second.loc[j, "StartDate"]) if j in links else None
        for j in second.index
    ]
    out["first_survey_on_child_form"] = [j in cross_form for j in second.index]

for key, df in processed.items():
    df.to_csv(PROCESSED / f"{key}.csv", index=False)
    print(f"{key}.csv  {len(df)} rows, {df.shape[1]} columns")

child_first.csv  20 rows, 20 columns
child_second.csv  10 rows, 24 columns
adult_first.csv  83 rows, 20 columns
adult_second.csv  39 rows, 24 columns


## Advisory board and qualitative counts

The board file keeps only the fields used in Supplemental Table 1. From each NVivo codebook the notebook keeps three numbers: the number of transcripts (the most files any code appears in), the number of codes, and the number of coded references.

In [11]:
board_out = pd.DataFrame({
    "age_band": board["demo_age"].map(age_band),
    "gender": board["demo_gender"],
    "race": board["demo_race_eth"].map(first_race),
    "race_more_than_one": board["demo_race_eth"].fillna("").astype(str).str.contains(","),
    "income": board["demo_income"],
})
board_out.to_csv(PROCESSED / "advisory_board.csv", index=False)

rows = []
for label, words in [("Participant interviews", ("codebook", "post-survey")),
                     ("Advisory board interviews", ("codebook", "1-on-1")),
                     ("Advisory board meetings", ("codebook", "meeting"))]:
    book = pd.read_excel(find_file(*words), sheet_name="Master Codebook").dropna(how="all")
    rows.append({"source": label, "transcripts": int(book["Files"].max()),
                 "codes": len(book), "coded_references": int(book["References"].sum())})
qualitative = pd.DataFrame(rows)
qualitative.to_csv(PROCESSED / "qualitative_counts.csv", index=False)
qualitative

,source,transcripts,codes,coded_references
0,Participant interviews,26,54,557
1,Advisory board interviews,9,69,379
2,Advisory board meetings,3,88,182


## Counts that needed the names

These can only be worked out while names are available, so they are saved as plain totals for the analysis notebook.

In [12]:
summary = {
    "unique_participants": n_unique,
    "completed_both_first_surveys": len(same_person),
    "parents_or_guardians": n_parents,
    "parents_with_two_children": n_two_children,
    "parents_also_adult_survey_identical_name": n_parent_adult,
    "advisory_board_members": len(board_keys),
    "advisory_board_in_participant_surveys": n_board_respondents,
    "age_range": {key: [int(a.min()), int(a.max())] for key, a in ages.items()},
}
(PROCESSED / "cohort_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "unique_participants": 101,
  "completed_both_first_surveys": 2,
  "parents_or_guardians": 17,
  "parents_with_two_children": 3,
  "parents_also_adult_survey_identical_name": 5,
  "advisory_board_members": 11,
  "advisory_board_in_participant_surveys": 7,
  "age_range": {
    "child_first": [
      5,
      14
    ],
    "child_second": [
      5,
      14
    ],
    "adult_first": [
      14,
      73
    ],
    "adult_second": [
      8,
      62
    ]
  }
}


## Last check

Search every processed file for any word from any name in the raw exports. The run stops if one turns up.

In [13]:
name_columns = {"child_first": ["child_name", "parent_name"], "child_second": ["child_name", "parent_name"],
                "adult_first": ["name"], "adult_second": ["name", "email_address"]}
words = set()
for key, cols in name_columns.items():
    for col in cols:
        for value in surveys[key][col].dropna():
            words |= {w for w in re.split(r"[^a-z]+", plain(value)) if len(w) >= 4}
for value in board["name"].dropna():
    words |= {w for w in re.split(r"[^a-z]+", plain(value)) if len(w) >= 4}

found = []
for path in PROCESSED.iterdir():
    text = plain(path.read_text())
    found += [(path.name, w) for w in words if re.search(rf"\b{w}\b", text)]

assert not found, f"possible names in processed files: {found}"
print(f"checked {len(words)} name words against {len(list(PROCESSED.iterdir()))} processed files: none found")

checked 224 name words against 7 processed files: none found
